In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "04-inference-engine/paged-attention")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# PagedAttention — practice notebook

Fill in the blanks (`...` / `# TODO`) in each exercise cell, delete that cell's
`raise NotImplementedError(...)` line, then run the **test cell** below it. It
prints `Exercise N passed` when your answer is right and stops with an
`AssertionError` when it is wrong. Until you attempt an exercise, running the
notebook stops there with `NotImplementedError`. The exercises build on each
other in the order they appear.

| Exercise | Concept |
|---|---|
| 1 | Ordinary attention over a contiguous KV cache (the baseline) |
| 2 | The block table: slot arithmetic + on-demand allocation |
| 3 | Paged attention as a gather through the block table |
| 4 | The real kernel: block-by-block **online softmax** |
| 5 | Forking + **copy-on-write** (parallel sampling) |

Answer key: `paged_attention_minimal.py` (same variable names, same logic —
peek per-exercise if stuck). Only dependency: numpy.

In [ ]:
import numpy as np

D_HEAD = 8        # head dimension (tiny, so arrays are easy to inspect)
BLOCK_SIZE = 4    # tokens per KV block (vLLM's default is 16)
NUM_BLOCKS = 64   # physical blocks in the pool
RNG = np.random.default_rng(0)

def softmax(x):
    x = x - x.max(axis=-1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=-1, keepdims=True)

# THE physical pool — one big pre-allocated region, like vLLM's KV cache
# tensor. K_POOL[b, s] = key vector in slot s of physical block b.
K_POOL = np.zeros((NUM_BLOCKS, BLOCK_SIZE, D_HEAD))
V_POOL = np.zeros_like(K_POOL)

class BlockManager:
    """Hands out physical blocks; refcounts enable sharing (Ex 5).
    Provided in full — it is plumbing, not the concept."""
    def __init__(self, num_blocks=NUM_BLOCKS):
        self.free = list(range(num_blocks))
        RNG.shuffle(self.free)      # so allocations *look* scattered
        self.refcount = {}
    def allocate(self):
        if not self.free:
            raise MemoryError("KV pool exhausted — vLLM would preempt here")
        b = self.free.pop()
        self.refcount[b] = 1
        return b
    def share(self, b):
        self.refcount[b] += 1
    def release(self, b):
        self.refcount[b] -= 1
        if self.refcount[b] == 0:
            del self.refcount[b]
            self.free.append(b)

print("setup ok")

## Exercise 1 — the baseline: attention over a contiguous cache

One decode step, one head: query `q` of shape `(d,)` attends over keys/values
`K, V` of shape `(t, d)` sitting in one contiguous array. Scaled dot-product:
scores = `K @ q / sqrt(d)`, weights = softmax(scores), output = weights `@ V`.

Everything paged must reproduce **exactly** this.

In [ ]:
def naive_attention(q, K, V):
    # YOUR CODE HERE: fill in each `...` below, then delete this raise line
    raise NotImplementedError("Exercise 1: naive_attention")
    scores = ...          # TODO: (t,) — dot each key with q, scale by sqrt(D_HEAD)
    weights = ...         # TODO: softmax over the t scores
    return ...            # TODO: (d,) — weight the values

In [ ]:
_r = np.random.default_rng(42)
_K, _V, _q = _r.normal(size=(6, D_HEAD)), _r.normal(size=(6, D_HEAD)), _r.normal(size=D_HEAD)
_out = naive_attention(_q, _K, _V)
assert _out.shape == (D_HEAD,)
assert np.allclose(_out, np.array([ 0.07764766492266414,  0.5122331607344466 , -0.23340202299789303,
  0.2999766737607332 , -0.8447176232265476 , -0.03192455594342712,
 -0.3289304579710528 , -0.2683423473354746 ])), "values off — check scaling and the softmax axis"
print("Exercise 1 passed — baseline attention works")

## Exercise 2 — the block table (the page table)

A `Sequence` owns **no tensors** — only `block_table`, a list mapping *logical*
block index → *physical* block id, plus a token count. `append_kv` stores one
token's K/V into the pool:

* the token lands in slot `num_tokens % BLOCK_SIZE` of the **last** logical block;
* when that slot is `0`, the previous block is full (or this is the first
  token) — allocate a fresh physical block **on demand**.

This on-demand growth is the entire memory-efficiency win: waste is at most
one partially-filled block per sequence. (Ignore sharing until Exercise 5.)

In [ ]:
class Sequence:
    def __init__(self, mgr):
        self.mgr = mgr
        self.block_table = []   # logical block index -> physical block id
        self.num_tokens = 0
        self.kv_log = []        # ground truth for tests ONLY — a real engine
                                # stores nothing outside the pool

    def append_kv(self, k, v):
        # YOUR CODE HERE: fill in each `...` below, then delete this raise line
        raise NotImplementedError("Exercise 2: Sequence.append_kv")
        slot = ...                          # TODO: slot inside current block (hint: %)
        if ...:                             # TODO: when do we need a new block?
            self.block_table.append(self.mgr.allocate())
        phys = ...                          # TODO: physical id of the LAST logical block
        K_POOL[phys, slot] = k
        V_POOL[phys, slot] = v
        self.num_tokens += 1
        self.kv_log.append((np.array(k), np.array(v)))

In [ ]:
_r = np.random.default_rng(1)
_mgr = BlockManager()
_seq = Sequence(_mgr)
for _ in range(10):
    _seq.append_kv(_r.normal(size=D_HEAD), _r.normal(size=D_HEAD))
assert _seq.num_tokens == 10
assert len(_seq.block_table) == 3, "10 tokens / block size 4 -> 3 blocks (4+4+2)"
assert len(set(_seq.block_table)) == 3, "each logical block needs its own physical block"
for _t in range(10):
    _b = _seq.block_table[_t // BLOCK_SIZE]
    assert np.allclose(K_POOL[_b, _t % BLOCK_SIZE], _seq.kv_log[_t][0]), \
        f"token {_t} not where the block table says it should be"
print(f"Exercise 2 passed — block table (logical->physical): {dict(enumerate(_seq.block_table))}")

## Exercise 3 — paged attention as a gather

Simplest correct version: rebuild contiguous `K, V` by following the block
table, trim to `num_tokens` (the last block may be partially filled with
stale zeros), and call your Exercise-1 attention.

The point to internalize: **the indirection changes where bytes live, not the
math**. The output must be bit-for-bit `allclose` with contiguous attention.

In [ ]:
def paged_attention_gather(q, seq):
    # YOUR CODE HERE: fill in each `...` below, then delete this raise line
    raise NotImplementedError("Exercise 3: paged_attention_gather")
    t = seq.num_tokens
    K = ...     # TODO: stack K_POOL rows of seq's physical blocks, in logical order, trim to t
    V = ...     # TODO: same for V_POOL
    return naive_attention(q, K, V)

In [ ]:
_r = np.random.default_rng(2)
_mgr = BlockManager()
_seq = Sequence(_mgr)
for _ in range(13):                       # crosses block boundaries; last block 1/4 full
    _seq.append_kv(_r.normal(size=D_HEAD), _r.normal(size=D_HEAD))
_q = _r.normal(size=D_HEAD)
_Kc = np.stack([k for k, _ in _seq.kv_log]); _Vc = np.stack([v for _, v in _seq.kv_log])
assert np.allclose(naive_attention(_q, _Kc, _Vc), paged_attention_gather(_q, _seq)), \
    "gather disagrees with contiguous — check ordering and the trim to t"
print("Exercise 3 passed — scattered blocks, identical attention output")

## Exercise 4 — the real kernel: block-by-block online softmax

A real GPU kernel never materializes the gathered `K, V`. It visits one
physical block at a time and folds it into a running accumulator — the same
online-softmax trick FlashAttention uses for tiling, applied here to a
follow-the-block-table access pattern.

State carried across blocks: running max `m` (numerical stability), running
softmax denominator `l`, running weighted value sum `acc`. For each block
with scores `s`:

$$m_{new} = \max(m, \max(s)) \qquad \alpha = e^{m - m_{new}} \qquad p = e^{s - m_{new}}$$
$$l \leftarrow \alpha\, l + \textstyle\sum p \qquad acc \leftarrow \alpha\, acc + p^\top V_b \qquad m \leftarrow m_{new}$$

Invariant after every block: `acc / l` equals attention over all tokens seen
so far. `α` rescales the old accumulator because the shared max just moved.

Why the *running* max and not just this block's `max(s)`: the algebra is exact
for any reference value, but the reference must never go down. If a later
block's scores sit far below an earlier block's, `α = e^{m - max(s)}` overflows
to `inf` and the result becomes `nan`. The check includes such a sequence.

In [ ]:
def paged_attention_blockwise(q, seq):
    # YOUR CODE HERE: fill in each `...` below, then delete this raise line
    raise NotImplementedError("Exercise 4: paged_attention_blockwise")
    m = float("-inf")        # running max of scores
    l = 0.0                  # running softmax denominator
    acc = np.zeros(D_HEAD)   # running weighted sum of values
    remaining = seq.num_tokens
    for phys in seq.block_table:
        n = min(BLOCK_SIZE, remaining)     # last block may be partial
        remaining -= n
        Kb, Vb = K_POOL[phys, :n], V_POOL[phys, :n]
        s = Kb @ q / np.sqrt(D_HEAD)       # this block's scores, shape (n,)
        m_new = ...                        # TODO: new running max
        alpha = ...                        # TODO: rescale factor for old m
        p = ...                            # TODO: this block's exp(s - m_new), shape (n,)
        l = ...                            # TODO: fold p into the denominator
        acc = ...                          # TODO: fold p @ Vb into the numerator
        m = m_new
    return acc / l

In [ ]:
_r = np.random.default_rng(3)
for _T in (3, 16, 37):                    # partial block, exact fit, many blocks
    _mgr = BlockManager()
    _seq = Sequence(_mgr)
    for _ in range(_T):
        _seq.append_kv(_r.normal(size=D_HEAD), _r.normal(size=D_HEAD))
    _q = _r.normal(size=D_HEAD)
    _Kc = np.stack([k for k, _ in _seq.kv_log]); _Vc = np.stack([v for _, v in _seq.kv_log])
    assert np.allclose(naive_attention(_q, _Kc, _Vc), paged_attention_blockwise(_q, _seq)), \
        f"blockwise disagrees at T={_T} — check the alpha rescaling of l and acc"
# The running max matters: block 0 scores about +800, block 1 about -800.
_mgr = BlockManager()
_seq = Sequence(_mgr)
_q = np.ones(D_HEAD)
for _t in range(2 * BLOCK_SIZE):
    _sign = 1.0 if _t < BLOCK_SIZE else -1.0
    _seq.append_kv(_sign * 800 / np.sqrt(D_HEAD) + _r.normal(size=D_HEAD), _r.normal(size=D_HEAD))
_Kc = np.stack([k for k, _ in _seq.kv_log]); _Vc = np.stack([v for _, v in _seq.kv_log])
with np.errstate(all="ignore"):
    _got = paged_attention_blockwise(_q, _seq)
assert np.all(np.isfinite(_got)) and np.allclose(naive_attention(_q, _Kc, _Vc), _got), \
    "a block scoring far below an earlier one broke it: m_new must be the RUNNING max, max(m, max(s))"
print("Exercise 4 passed — online softmax matches, no gathered tensor needed")

## Exercise 5 — fork + copy-on-write (parallel sampling)

Sampling *n* completions from one prompt should not copy the prompt's KV
cache *n* times. With a block table it doesn't have to:

* `fork` gives the child the **same physical block ids** and bumps each
  block's refcount (`mgr.share`);
* nobody copies anything until a sequence **writes into a shared block** —
  then it allocates a fresh block, copies the contents, releases its shared
  reference, and remaps its own block table. Copy-on-write, at block
  granularity.

Below, paste your working Exercise-2 lines where marked, then add the CoW
branch. The check also audits the pool: every block's refcount must equal the
number of sequences holding it, and freeing all three sequences must return
every block. A copy that never releases the shared block leaks it.

In [ ]:
def fork(seq):
    # YOUR CODE HERE: fill in each `...` below, then delete this raise line
    raise NotImplementedError("Exercise 5: fork")
    child = Sequence(seq.mgr)
    child.block_table = ...             # TODO: SAME physical ids as the parent
    child.num_tokens = seq.num_tokens
    child.kv_log = list(seq.kv_log)
    for b in child.block_table:
        ...                             # TODO: tell the manager this block now has one more owner
    return child

class Sequence(Sequence):               # extends Ex-2 Sequence with CoW
    def append_kv(self, k, v):
        # YOUR CODE HERE: fill in each `...` below, then delete this raise line
        raise NotImplementedError("Exercise 5: copy-on-write in append_kv")
        slot = ...                      # TODO: paste your Exercise-2 answer
        if ...:                         # TODO: paste your Exercise-2 answer
            self.block_table.append(self.mgr.allocate())
        else:
            phys = self.block_table[-1]
            if self.mgr.refcount[phys] > 1:     # shared! copy-on-write:
                fresh = ...             # TODO: get a private block
                ...                     # TODO: copy K_POOL and V_POOL contents phys -> fresh
                ...                     # TODO: release the shared block; remap block_table[-1]
        phys = self.block_table[-1]
        K_POOL[phys, slot] = k
        V_POOL[phys, slot] = v
        self.num_tokens += 1
        self.kv_log.append((np.array(k), np.array(v)))

In [ ]:
_r = np.random.default_rng(4)
_mgr = BlockManager()
_parent = Sequence(_mgr)
for _ in range(6):                        # 6-token prompt: 1 full + 1 half-full block
    _parent.append_kv(_r.normal(size=D_HEAD), _r.normal(size=D_HEAD))
_samples = [_parent, fork(_parent), fork(_parent)]
assert [_mgr.refcount[b] for b in _parent.block_table] == [3, 3], \
    "after two forks every prompt block should have refcount 3"
for _s in _samples:                       # each sample generates 3 of its own tokens
    for _ in range(3):
        _s.append_kv(_r.normal(size=D_HEAD), _r.normal(size=D_HEAD))
_first = {_s.block_table[0] for _s in _samples}
_last = [_s.block_table[1] for _s in _samples]
assert len(_first) == 1, "the full prompt block should STILL be shared by all 3"
assert len(set(_last)) == 3, "the half-full block should have been copy-on-written"
_q = _r.normal(size=D_HEAD)
for _s in _samples:                       # divergence didn't corrupt anyone's history
    _Kc = np.stack([k for k, _ in _s.kv_log]); _Vc = np.stack([v for _, v in _s.kv_log])
    assert np.allclose(naive_attention(_q, _Kc, _Vc), paged_attention_blockwise(_q, _s))
_unique = {b for _s in _samples for b in _s.block_table}
_owners = {}                              # block id -> how many sequences hold it
for _s in _samples:
    for _b in _s.block_table:
        _owners[_b] = _owners.get(_b, 0) + 1
assert _mgr.refcount == _owners, \
    "a refcount no longer matches the sequences holding the block -> the CoW must release the shared block"
assert len(_mgr.free) == NUM_BLOCKS - len(_unique), "blocks leaked: in use but held by no sequence"
for _s in _samples:                       # finish all three: every block must come back
    for _b in _s.block_table:
        _mgr.release(_b)
assert len(_mgr.free) == NUM_BLOCKS and not _mgr.refcount, "after freeing every sequence the pool is not full again"
print(f"Exercise 5 passed — {len(_unique)} physical blocks serve 3 sequences "
      f"(9 without sharing); all outputs still exact")

## Wrap-up — questions worth answering out loud

You have now built every load-bearing piece of PagedAttention: block tables,
on-demand allocation, a block-walking kernel with online softmax, and
refcounted copy-on-write. Stretch yourself:

1. **Block size.** Set `BLOCK_SIZE = 1`, rerun everything, then `16`. What
   improves and what gets worse in each direction? (Think: per-sequence
   waste, block-table length, sharing granularity, and — on a real GPU —
   memory coalescing in the kernel.)
2. **Preemption.** The `MemoryError` in `allocate` is where vLLM preempts.
   What extra state would *swap-to-CPU* need in this code? Why is
   *free-and-recompute* often faster in practice, and why is eviction
   all-or-nothing per sequence?
3. **GQA / MLA.** Which axis would you add to `K_POOL` for `n_kv_heads`, and
   why does grouped-query attention shrink the pool without touching any of
   the paging logic you wrote?
4. **Prefix caching.** Exercise 5 shares blocks *within* one request's
   samples. What lookup structure would let two *different* requests with the
   same system prompt share blocks? (That's SGLang's RadixAttention.)

Answer key with runnable demos: `paged_attention_minimal.py`.